In [ ]:
import subprocess, time, os
from langchain_mcp_adapters.client import MultiServerMCPClient
from chemgraph.agent.llm_agent import ChemGraph

prompt_single = "Run a mace calculations with the same file, use energy for driver and small model. a cif file are located at /Users/hari/projects/ChemGraph/notebooks/cif_files/calf-20_pacmof.cif"

os.environ["ALCF_ACCESS_TOKEN"]="<your alcf inference service token. Obtain one from here: https://docs.alcf.anl.gov/services/inference-endpoints/"

try:
    client = MultiServerMCPClient({
        "Chemistry Tools MCP": {
            "command": "python",
            "args": ["-m", "chemgraph.mcp.mace_mcp_hpc"],
    	    "transport": "stdio",
            },
        }
    )
    
    tools = await client.get_tools()
    
    cg_single = ChemGraph(
        model_name="openai/gpt-oss-120b",
        base_url="https://inference-api.alcf.anl.gov/resource_server/sophia/vllm/v1",
        structured_output=False,
        return_option="state",
        tools=tools,
    )
    result_single = await cg_single.run(prompt_single, config={"configurable": {"thread_id": "1"}})
finally:
    print("Done")

Done creating client


2026-05-22 12:34:00,370 - chemgraph.graphs.single_agent - INFO - Constructing single agent graph
2026-05-22 12:34:00,372 - chemgraph.graphs.single_agent - INFO - Graph construction completed


Done getting tools
================================ Human Message =================================

Run a mace calculations with the same file, use energy for driver and small model. a cif file are located at /Users/hari/projects/ChemGraph/notebooks/cif_files/calf-20_pacmof.cif
================================== Ai Message ==================================
Tool Calls:
  run_mace_single (chatcmpl-tool-a42c48d32a55e54d)
 Call ID: chatcmpl-tool-a42c48d32a55e54d
  Args:
    params: {'input_structure_file': '/Users/hari/projects/ChemGraph/notebooks/cif_files/calf-20_pacmof.cif', 'driver': 'energy', 'model': 'small'}
================================= Tool Message =================================
Name: run_mace_single

{
  "status": "success",
  "message": "Simulation completed. Results saved to /Users/hari/projects/ChemGraph/notebooks/output.json",
  "single_point_energy": -295.75144320599975,
  "unit": "eV"
}
================================== Ai Message =================================

## Client-Only Mode: Shared EL Orchestrator

Start an EnsembleLauncher orchestrator in this notebook, then connect MCP servers to it in **client-only mode**. Multiple MCP servers can share the same orchestrator — each connects via `ClusterClient` using the orchestrator's `checkpoint_dir`.

In [ ]:
import os, time, toml, random
from ensemble_launcher import EnsembleLauncher
from ensemble_launcher.config import LauncherConfig, MPIConfig, PolicyConfig, SystemConfig
from ensemble_launcher.helper_functions import get_nodes
from langchain_mcp_adapters.client import MultiServerMCPClient
from chemgraph.agent.llm_agent import ChemGraph

# --- 1. Start the EL orchestrator in this notebook ---
system_config = SystemConfig(
    name="local",
    ncpus=os.cpu_count(),
    cpus=list(range(os.cpu_count())),
)

launcher_config = LauncherConfig(
    child_executor_name="async_mpi",
    task_executor_name="async_processpool",
    return_stdout=True,
    worker_logs=True,
    master_logs=True,
    children_scheduler_policy="fixed_leafs_children_policy",
    policy_config=PolicyConfig(nlevels=2, leaf_nodes=len(get_nodes())),
    cluster=True,
    checkpoint_dir=os.path.join(os.getcwd(), f".ckpt_notebook_demo_{random.randint(0,10000)}"),
    mpi_config=MPIConfig(flavor="test"),
)

os.makedirs(launcher_config.checkpoint_dir, exist_ok=True)
orchestrator = EnsembleLauncher(
    ensemble_file={},
    system_config=system_config,
    launcher_config=launcher_config,
)
orchestrator.start()
time.sleep(10.0)
print(f"EL orchestrator running — checkpoint_dir: {launcher_config.checkpoint_dir}")

os.environ["ALCF_ACCESS_TOKEN"]="<your alcf inference service token. Obtain one from here: https://docs.alcf.anl.gov/services/inference-endpoints/"

# --- 2. Update config.toml for client-only mode ---
config_path = os.path.join(os.path.dirname(os.getcwd()), "config.toml")
cfg = toml.load(config_path)
cfg["execution"] = {
    "backend": "ensemble_launcher",
    "system": "local",
    "ensemble_launcher": {
        "client_only": True,
        "checkpoint_dir": launcher_config.checkpoint_dir,
    },
}
with open(config_path, "w") as f:
    toml.dump(cfg, f)

# --- 3. Start MCP server in client-only mode ---
prompt = (
    "Run a mace calculation on the structure at "
    "/Users/hari/projects/ChemGraph/notebooks/cif_files/calf-20_pacmof.cif "
    "using energy driver and small model."
)

try:
    client = MultiServerMCPClient({
        "MACE MCP (EL client-only)": {
            "command": "python",
            "args": ["-m", "chemgraph.mcp.mace_mcp_hpc"],
            "transport": "stdio",
        },
    })
    tools = await client.get_tools()
    print(f"Tools available: {[t.name for t in tools]}")

    cg_single = ChemGraph(
            model_name="openai/gpt-oss-120b",
            base_url="https://inference-api.alcf.anl.gov/resource_server/sophia/vllm/v1",
            structured_output=False,
            return_option="state",
            tools=tools,
        )
    result = await cg_single.run(prompt, config={"configurable": {"thread_id": "el-client"}})
finally:
    orchestrator.stop()
    print("Done — orchestrator stopped.")

EL orchestrator running — checkpoint_dir: /Users/hari/projects/ChemGraph/notebooks/.ckpt_notebook_demo_6514


2026-05-22 12:38:21,496 - chemgraph.graphs.single_agent - INFO - Constructing single agent graph
2026-05-22 12:38:21,501 - chemgraph.graphs.single_agent - INFO - Graph construction completed


Tools available: ['run_mace_single', 'run_mace_ensemble', 'extract_output_json', 'check_job_status', 'get_job_results', 'list_jobs', 'cancel_job', 'check_endpoint_status']
================================ Human Message =================================

Run a mace calculation on the structure at /Users/hari/projects/ChemGraph/notebooks/cif_files/calf-20_pacmof.cif using energy driver and small model.
================================== Ai Message ==================================
Tool Calls:
  run_mace_single (chatcmpl-tool-b437a126aeb2f6ba)
 Call ID: chatcmpl-tool-b437a126aeb2f6ba
  Args:
    params: {'input_structure_file': '/Users/hari/projects/ChemGraph/notebooks/cif_files/calf-20_pacmof.cif', 'driver': 'energy', 'model': 'small'}


================================= Tool Message =================================
Name: run_mace_single

{
  "status": "success",
  "message": "Simulation completed. Results saved to /Users/hari/projects/ChemGraph/notebooks/output.json",
  "single_point_energy": -295.75144320599975,
  "unit": "eV"
}
================================== Ai Message ==================================

The MACE single‑point energy calculation completed successfully.

**Result**

| Item                     | Value                                   |
|--------------------------|-----------------------------------------|
| **Status**               | success                                 |
| **Message**              | Simulation completed. Results saved to `/Users/hari/projects/ChemGraph/notebooks/output.json` |
| **Single‑point energy**  | **‑295.75144320599975 eV**              |
| **Unit**                 | eV                                      |

The output JSON file (`/Users/hari/projects/ChemGraph/noteb

In [3]:
prompt_multi = "You are given a chemical reaction: 1 (Methane) + 2 (Oxygen) -> 1 (Carbon dioxide) + 2 (Water). Calculate the enthalpy for this reaction using MACE MP at 400K."

client = MultiServerMCPClient({
    "Chemistry Tools MCP": {
        "command": "python",
        "args": ["-m", "chemgraph.mcp.mcp_tools"],
    	"transport": "stdio",
    },
}
)
tools = await client.get_tools()
cg_multi = ChemGraph(
    model_name="gpt-4.1",
    workflow_type="multi_agent",
    structured_output=False,
    return_option="state",
    tools=tools,
)
result_multi = await cg_multi.run(prompt_multi, config={"configurable": {"thread_id": "2"}})

2026-01-22 13:50:43,296 - chemgraph.models.openai - INFO - Loading OpenAI model: gpt-4.1


INFO:chemgraph.models.openai:Loading OpenAI model: gpt-4.1


2026-01-22 13:50:43,297 - chemgraph.models.openai - INFO - Requested model: gpt-4.1


INFO:chemgraph.models.openai:Requested model: gpt-4.1


2026-01-22 13:50:43,298 - chemgraph.models.openai - INFO - OpenAI model loaded successfully


...

BFGS:    5 13:51:02       -9.381692        0.041391
BFGS:    6 13:51:02       -9.381704        0.000854
Enthalpy components at T = 400.00 K:
E_pot                 -9.382 eV
E_ZPE                  0.095 eV
Cv_trans (0->T)        0.052 eV
Cv_rot (0->T)          0.034 eV
Cv_vib (0->T)          0.001 eV
(C_v -> C_p)           0.034 eV
-------------------------------
H                     -9.165 eV
Entropy components at T = 400.00 K and P = 101325.0 Pa:
                           S               T*S
S_trans (1 bar)    0.0016395 eV/K        0.656 eV
S_rot              0.0004836 eV/K        0.193 eV
S_elec             0.0000000 eV/K        0.000 eV
S_vib              0.0000023 eV/K        0.001 eV
S (1 bar -> P)    -0.0000011 eV/K       -0.000 eV
-------------------------------------------------
S                  0.0021242 eV/K        0.850 eV
Enthalpy components at T = 400.00 K:
E_pot                 -9.382 eV
E_ZPE                  0.095 eV
Cv_trans (0->T)        0.052 eV
Cv_rot (0->T)    

/Users/tpham2/work/projects/ChemGraph/env/chemgraph_env/lib/python3.10/site-packages/mace/calculators/mace.py:197: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


Enthalpy components at T = 400.00 K:
E_pot                -22.546 eV
E_ZPE                  0.318 eV
Cv_trans (0->T)        0.052 eV
Cv_rot (0->T)          0.034 eV
Cv_vib (0->T)          0.015 eV
(C_v -> C_p)           0.034 eV
-------------------------------
H                    -22.092 eV
Entropy components at T = 400.00 K and P = 101325.0 Pa:
                           S               T*S
S_trans (1 bar)    0.0016807 eV/K        0.672 eV
S_rot              0.0005947 eV/K        0.238 eV
S_elec             0.0000000 eV/K        0.000 eV
S_vib              0.0000507 eV/K        0.020 eV
S (1 bar -> P)    -0.0000011 eV/K       -0.000 eV
-------------------------------------------------
S                  0.0023250 eV/K        0.930 eV
Enthalpy components at T = 400.00 K:
E_pot                -22.546 eV
E_ZPE                  0.318 eV
Cv_trans (0->T)        0.052 eV
Cv_rot (0->T)          0.034 eV
Cv_vib (0->T)          0.015 eV
(C_v -> C_p)           0.034 eV
-------------------------

/Users/tpham2/work/projects/ChemGraph/env/chemgraph_env/lib/python3.10/site-packages/mace/calculators/mace.py:197: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


BFGS:    4 13:51:15      -13.786069        0.024777
BFGS:    5 13:51:15      -13.786082        0.004130
Enthalpy components at T = 400.00 K:
E_pot                -13.786 eV
E_ZPE                  0.577 eV
Cv_trans (0->T)        0.052 eV
Cv_rot (0->T)          0.052 eV
Cv_vib (0->T)          0.001 eV
(C_v -> C_p)           0.034 eV
-------------------------------
H                    -13.071 eV
Entropy components at T = 400.00 K and P = 101325.0 Pa:
                           S               T*S
S_trans (1 bar)    0.0015652 eV/K        0.626 eV
S_rot              0.0004947 eV/K        0.198 eV
S_elec             0.0000000 eV/K        0.000 eV
S_vib              0.0000015 eV/K        0.001 eV
S (1 bar -> P)    -0.0000011 eV/K       -0.000 eV
-------------------------------------------------
S                  0.0020603 eV/K        0.824 eV
Enthalpy components at T = 400.00 K:
E_pot                -13.786 eV
E_ZPE                  0.577 eV
Cv_trans (0->T)        0.052 eV
Cv_rot (0->T)    